# Book Recommendation System
Modified version using the popular Kaggle Book Recommendation Dataset

## 1. Install Dependencies

In [1]:
# Install all dependencies from requirements.txt
%pip install -q -r requirements.txt

# Download the dataset from Kaggle
! kaggle datasets download -d arashnic/book-recommendation-dataset
! unzip -o book-recommendation-dataset.zip -d data/

print("✓ Dataset files should be in data/ directory:")
print("  - Books.csv")
print("  - Ratings.csv")
print("  - Users.csv")

Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset
License(s): CC0-1.0
book-recommendation-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  book-recommendation-dataset.zip
  inflating: data/Books.csv          
  inflating: data/DeepRec.png        
  inflating: data/Ratings.csv        
  inflating: data/Users.csv          
  inflating: data/classicRec.png     
  inflating: data/recsys_taxonomy2.png  
✓ Dataset files should be in data/ directory:
  - Books.csv
  - Ratings.csv
  - Users.csv


## 2. Setup and Imports

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")  # Adjust the path to your .env file


print("✓ Base imports successful")

✓ Base imports successful


## 3. Load and Explore Real Dataset from Kaggle

In [3]:
# Load the Kaggle datasets
print("Loading Kaggle Book Recommendation Dataset...\n")

books_df = pd.read_csv('data/Books.csv', encoding='latin-1')
ratings_df = pd.read_csv('data/Ratings.csv')
users_df = pd.read_csv('data/Users.csv')

print(f"Books dataset: {books_df.shape}")
print(f"Ratings dataset: {ratings_df.shape}")
print(f"Users dataset: {users_df.shape}")

# Display sample data
print("\n=== Books Dataset Sample ===")
print(books_df.head(3))

print("\n=== Ratings Dataset Sample ===")
print(ratings_df.head(3))

print("\n=== Books Dataset Columns ===")
print(books_df.columns.tolist())

Loading Kaggle Book Recommendation Dataset...



/tmp/ipykernel_4059/2395822538.py:4: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv('data/Books.csv', encoding='latin-1')


Books dataset: (271360, 8)
Ratings dataset: (1149780, 3)
Users dataset: (278858, 3)

=== Books Dataset Sample ===
         ISBN            Book-Title           Book-Author Year-Of-Publication  \
0  0195153448   Classical Mythology    Mark P. O. Morford                2002   
1  0002005018          Clara Callan  Richard Bruce Wright                2001   
2  0060973129  Decision in Normandy          Carlo D'Este                1991   

                 Publisher                                        Image-URL-S  \
0  Oxford University Press  http://images.amazon.com/images/P/0195153448.0...   
1    HarperFlamingo Canada  http://images.amazon.com/images/P/0002005018.0...   
2          HarperPerennial  http://images.amazon.com/images/P/0060973129.0...   

                                         Image-URL-M  \
0  http://images.amazon.com/images/P/0195153448.0...   
1  http://images.amazon.com/images/P/0002005018.0...   
2  http://images.amazon.com/images/P/0060973129.0...   

           

## 4. Prepare Data for LangChain (Sample for Speed)

In [4]:
# Sample the data to make processing faster
# The full dataset has 271K books - we'll use top books by ratings

# Merge ratings with books
#merged = ratings_df.merge(books_df, on='ISBN', how='inner')

# Get top books by number of ratings
top_books_isbn = ratings_df.groupby('ISBN').size().nlargest(1000).index
print(f"Selecting top 1000 books by rating count...")

# Filter to top books
books_sample = books_df[books_df['ISBN'].isin(top_books_isbn)].copy()

# Rename columns to be more user-friendly
books_sample.rename(columns={
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'ISBN': 'isbn'
}, inplace=True)

# Save sampled dataset for document loader
books_sample.to_csv('books_for_llm.csv', index=False)

print(f"✓ Sampled dataset: {books_sample.shape[0]} books")
print(f"\nDataset Preview:")
print(books_sample[['title', 'author', 'year', 'publisher', 'isbn']].head(10))

Selecting top 1000 books by rating count...
✓ Sampled dataset: 992 books

Dataset Preview:
                                   title              author  year  \
18                         The Testament        John Grisham  1999   
19  Beloved (Plume Contemporary Fiction)       Toni Morrison  1994   
26                           Wild Animus        Rich Shapero  2004   
27                              Airframe    Michael Crichton  1997   
28                              Timeline    MICHAEL CRICHTON  2000   
37                 To Kill a Mockingbird          Harper Lee  1988   
38        Seabiscuit: An American Legend   LAURA HILLENBRAND  2002   
45                    I'll Be Seeing You  Mary Higgins Clark  1994   
46            From the Corner of His Eye         Dean Koontz  2001   
47                          Isle of Dogs   Patricia Cornwell  2002   

                     publisher        isbn  
18                        Dell  0440234743  
19                       Plume  0452264464  
26 

## 5. Model Initialization (Model Agnostic)

In [5]:
from langchain.chat_models import init_chat_model
import huggingface_hub
import os

# Fix 401: login to HuggingFace Hub (only needed when using HF models)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
if HF_TOKEN:
    huggingface_hub.login(token=HF_TOKEN)

# Model-agnostic initialization — swap the active line to change provider
# OpenAI:        model="openai:gpt-4o-mini"
# Google Gemini: model="google_genai:gemini-2.5-flash-lite"
# HuggingFace lightweight models (no gating, runs on HF servers):
#   1.1B → "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"   ← lightest
#   3.8B → "huggingface:microsoft/Phi-3-mini-4k-instruct"
#   7B   → "huggingface:HuggingFaceH4/zephyr-7b-beta"

llm = init_chat_model(
    #model="openai:gpt-4o-mini",
    #model="google_genai:gemini-2.5-flash-lite",
    #model="huggingface:HuggingFaceH4/zephyr-7b-beta",
    #model="huggingface:microsoft/Phi-3-mini-4k-instruct",
    model="huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    temperature=0.7,
)

print("✓ LLM initialized successfully")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ LLM initialized successfully


## 6. Messages - Simple Conversation

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

# Messages provide a structured way to manage conversations
messages = [
    SystemMessage(content="You are a knowledgeable book recommendation assistant with expertise in literature."),
    HumanMessage(content="Recommed some classic literature books for me to read.")
]

response = llm.invoke(messages)
print("Assistant:", response.content[:500])
print("\n✓ Message-based conversation working")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Assistant: <|system|>
You are a knowledgeable book recommendation assistant with expertise in literature.</s>
<|user|>
Recommed some classic literature books for me to read.</s>
<|assistant|>
Sure, here are some classic literature books that you might enjoy:

1. "On the Road" by Jack Kerouac
2. "The Great Gatsby" by F. Scott Fitzgerald
3. "To Kill a Mockingbird" by Harper Lee
4. "The Catcher in the Rye" by J.D. Salinger
5. "Little Women" by Louisa May Alcott
6. "The Adventures of Huckleberry Finn" by Mark 

✓ Message-based conversation working


## 7. Document Loader - Load Books Sample Data

In [7]:
from langchain_community.document_loaders import CSVLoader

# Load books sample with LangChain CSVLoader (creates Document objects directly)
#  Each document represents one row of the CSV file
loader = CSVLoader(file_path="books_for_llm.csv")
documents = loader.load()

print(f"✓ Loaded {len(documents)} book documents for LLM context")
print(f"\nFirst book document:")
print(documents[0].page_content)

✓ Loaded 992 book documents for LLM context

First book document:
isbn: 0440234743
title: The Testament
author: John Grisham
year: 1999
publisher: Dell
Image-URL-S: http://images.amazon.com/images/P/0440234743.01.THUMBZZZ.jpg
Image-URL-M: http://images.amazon.com/images/P/0440234743.01.MZZZZZZZ.jpg
Image-URL-L: http://images.amazon.com/images/P/0440234743.01.LZZZZZZZ.jpg


## 8. Qdrant Vector Store — Insert & Retrieve

In [8]:
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

QDRANT_URL = os.environ.get("QDRANT_HOST")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")

# Lightweight embedding model (runs locally, free, no API key needed)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Insert all book documents into Qdrant (run once to populate the collection)
vectorstore = QdrantVectorStore.from_documents(
    documents,
    embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="books",
)

# Retriever: fetch top 5 most semantically relevant books per query
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Helper to format retrieved docs into a single string for the prompt
def format_books(docs):
    return "\n\n".join([doc.page_content for doc in docs])

print(f"✓ {len(documents)} books inserted into Qdrant")
print("✓ Retriever ready — fetches top 5 relevant books per query (not all 992)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ 992 books inserted into Qdrant
✓ Retriever ready — fetches top 5 relevant books per query (not all 992)


## 9. Retrieve & Inject into Prompt (RAG)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Retrieve top 5 books relevant to a sample query
sample_docs = retriever.invoke("fiction books from the 1990s")
print(f"Retrieved {len(sample_docs)} relevant books:\n")
for doc in sample_docs:
    print(doc.page_content[:120])
    print("---")

# RAG prompt — receives only retrieved context, not all 992 books
recommendation_prompt = ChatPromptTemplate.from_template(
    "You are a book recommendation expert.\n"
    "Relevant books from catalog:\n{context}\n\n"
    "User request: {user_query}\n"
    "Recommend books from the catalog above."
)

print("\n✓ Only top-5 retrieved books injected into prompt (not all 330K chars)")

Retrieved 5 relevant books:

isbn: 0316781266
title: The Last Time They Met : A Novel
author: Anita Shreve
year: 2002
publisher: Back Bay Books
Image
---
isbn: 0375412530
title: Back When We Were Grownups: A Novel
author: ANNE TYLER
year: 2001
publisher: Knopf
Image-URL-S: 
---
isbn: 0316781142
title: The Last Time They Met : A Novel
author: Anita Shreve
year: 2001
publisher: Little, Brown
Image-
---
isbn: 0156006529
title: Where or When  : A Novel
author: Anita Shreve
year: 1999
publisher: Harvest Books
Image-URL-S: h
---
isbn: 0449911519
title: Secret History : A Novel
author: DONNA TARTT
year: 1996
publisher: Ballantine Books
Image-URL-S:
---

✓ Only top-5 retrieved books injected into prompt (not all 330K chars)


## 9. Basic LangChain Expression Language (LCEL) Chaining

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# RAG chain: user query → Qdrant retrieves top 5 books → LLM answers
rag_chain = (
    {"context": retriever | format_books, "user_query": lambda x: x}
    | recommendation_prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke("I want books by popular American authors from the 1980s")
print("RAG Chain Output:")
print(result)

## 10. Control Flow - RunnableBranch

In [11]:
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

recommendation_chain = (
    {"context": retriever | format_books, "user_query": RunnablePassthrough()}
    | recommendation_prompt
    | llm
    | StrOutputParser()
)

analysis_prompt = ChatPromptTemplate.from_template(
    "You are a book expert.\n"
    "Relevant books:\n{context}\n\n"
    "Analyze/discuss: {user_query}"
)
analysis_chain = (
    {"context": retriever | format_books, "user_query": RunnablePassthrough()}
    | analysis_prompt
    | llm
    | StrOutputParser()
)

# Branches now take a plain string query (RunnablePassthrough handles routing)
router = RunnableBranch(
    (lambda x: "recommend" in x.lower(), recommendation_chain),
    (lambda x: "compare" in x.lower() or "analysis" in x.lower(), analysis_chain),
    recommendation_chain  # default
)

print("Test 1 - Recommend query:")
result1 = router.invoke("recommend a mystery thriller book")
print(result1[:300] + "...\n")

print("Test 2 - Analysis query:")
result2 = router.invoke("analyze trends in literature from 1990s")
print(result2[:300] + "...")
print("\n✓ RunnableBranch working")

Test 1 - Recommend query:
<|user|>
You are a book recommendation expert.
Relevant books from catalog:
isbn: 0553571885
title: A Thin Dark Line (Mysteries &amp; Horror)
author: TAMI HOAG
year: 1998
publisher: Bantam
Image-URL-S: http://images.amazon.com/images/P/0553571885.01.THUMBZZZ.jpg
Image-URL-M: http://images.amazon.com...

Test 2 - Analysis query:
<|user|>
You are a book recommendation expert.
Relevant books from catalog:
isbn: 0156006529
title: Where or When  : A Novel
author: Anita Shreve
year: 1999
publisher: Harvest Books
Image-URL-S: http://images.amazon.com/images/P/0156006529.01.THUMBZZZ.jpg
Image-URL-M: http://images.amazon.com/images...

✓ RunnableBranch working


## 11. Generate-Refine Pipeline

In [12]:
from langchain_core.runnables import RunnablePassthrough

recommend_chain = (
    {"context": retriever | format_books, "user_query": RunnablePassthrough()}
    | recommendation_prompt
    | llm
    | StrOutputParser()
)

refine_chain = (
    ChatPromptTemplate.from_template(
        "Here are some book recommendations:\n{recommendation}\n\n"
        "Please refine these recommendations by:\n"
        "1. Making them more concise\n"
        "2. Adding specific reasons why each book is perfect\n"
        "3. Mentioning the publication year\n"
        "4. Formatting them clearly"
    )
    | llm
    | StrOutputParser()
)

generate_refine_chain = (
    recommend_chain
    | (lambda x: {"recommendation": x})
    | refine_chain
)

final_recommendation = generate_refine_chain.invoke("I like literary fiction and historical novels")
print("Refined Recommendation:")
print(final_recommendation)

Refined Recommendation:
<|user|>
Here are some book recommendations:
<|user|>
You are a book recommendation expert.
Relevant books from catalog:
isbn: 0743235150
title: Everything's Eventual : 14 Dark Tales
author: Stephen King
year: 2002
publisher: Scribner
Image-URL-S: http://images.amazon.com/images/P/0743235150.01.THUMBZZZ.jpg
Image-URL-M: http://images.amazon.com/images/P/0743235150.01.MZZZZZZZ.jpg
Image-URL-L: http://images.amazon.com/images/P/0743235150.01.LZZZZZZZ.jpg

isbn: 0374199698
title: Middlesex: A Novel
author: Jeffrey Eugenides
year: 2002
publisher: Farrar, Straus and Giroux
Image-URL-S: http://images.amazon.com/images/P/0374199698.01.THUMBZZZ.jpg
Image-URL-M: http://images.amazon.com/images/P/0374199698.01.MZZZZZZZ.jpg
Image-URL-L: http://images.amazon.com/images/P/0374199698.01.LZZZZZZZ.jpg

isbn: 0553580930
title: A Man in Full
author: Tom Wolfe
year: 1999
publisher: Bantam
Image-URL-S: http://images.amazon.com/images/P/0553580930.01.THUMBZZZ.jpg
Image-URL-M: http:/

## 12. Message History Memory - Multi-turn Conversations

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableLambda

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Inject retrieved context dynamically per turn
def retrieve_context(inputs):
    docs = retriever.invoke(inputs["user_query"])
    inputs["context"] = format_books(docs)
    return inputs

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a book recommendation assistant.\nRelevant books from catalog:\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_query}")
])

conversational_chain = (
    RunnableLambda(retrieve_context)
    | conversational_prompt
    | llm
    | StrOutputParser()
)

memory_chain = RunnableWithMessageHistory(
    conversational_chain,
    get_session_history,
    input_messages_key="user_query",
    history_messages_key="history"
)

session_id = "user_session_001"

print("=== Turn 1 ===")
response1 = memory_chain.invoke(
    {"user_query": "I enjoy classic literature. Can you recommend something?"},
    config={"configurable": {"session_id": session_id}}
)
print(response1[:300] + "...")

print("\n=== Turn 2 (Bot remembers context) ===")
response2 = memory_chain.invoke(
    {"user_query": "Who wrote the first book you recommended?"},
    config={"configurable": {"session_id": session_id}}
)
print(response2[:300] + "...")

print("\n=== Full Conversation History ===")
history = get_session_history(session_id)
for i, msg in enumerate(history.messages):
    print(f"{i+1}. [{msg.type.upper()}]: {msg.content[:80]}...")

=== Turn 1 ===


/home/sirkumar/python3.14_venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 13. Serialization - Save and Load Chains

In [ ]:
from langchain_core.load import dumps, loads

# Serialize the shared recommendation_prompt from Section 8
serialized = dumps(recommendation_prompt)
print("Serialized Prompt (JSON):")
print(serialized[:500] + "...")

# Save to file
with open("book_chain_kaggle.json", "w") as f:
    f.write(serialized)
print("\n✓ Prompt saved to 'book_chain_kaggle.json'")

# Load the prompt from file
with open("book_chain_kaggle.json", "r") as f:
    loaded_serialized = f.read()

loaded_prompt = loads(loaded_serialized, allowed_objects="all")
print("✓ Prompt loaded from file")

# Rebuild the executable chain with the active llm in memory
loaded_chain = loaded_prompt | llm | StrOutputParser()

# Test the loaded chain (hardcoded catalog is already inside recommendation_prompt)
result = loaded_chain.invoke({
    "user_query": "fantasy and adventure novels"
})
print(f"\nLoaded Chain Output: {result[:300]}...")
